In [1]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

Go to root directory

In [2]:
if Path.cwd().name == "_utils":
    os.chdir(Path.cwd().parent)

print(f"Current Directory: {Path.cwd()}")

Current Directory: /work/tuv89272/Calibration_Pipeline_v7_NG-SE


In [3]:
# Read rasters
input_population_raster, metadata = read_raster(f"{generated_data_path}/{country_code}_population_backwards_projected_{initial_year}_20.36M.asc")

nodata = metadata["NODATA_value"]
valid_mask = input_population_raster != nodata
input_population_raster_valid = input_population_raster[valid_mask]

input_population = np.sum(input_population_raster_valid)
print(f"Sum of valid input population raster: {input_population:,.0f}")

Sum of valid input population raster: 20,357,570


In [4]:
# target_population = 21_515_594 # Scaling by PfPR Validation Population Difference 2021
# target_population = 20_636_083 # Scaling by Population Validation Population Difference 2024
target_population = 21_118_098 # Scaled by pop difference sim v observed 2020 with new death rate

target_raster_population_name = f"{target_population / 1_000_000:.2f}M"
print(f"Target population: {target_population:,.0f} ({target_raster_population_name})")
target_raster_name = f"{country_code}_population_backwards_projected_{initial_year}_{target_raster_population_name}.asc"
info(f"Target raster name: {target_raster_name}")

Target population: 21,118,098 (21.12M)
→ Target raster name: ng-se_population_backwards_projected_2011_21.12M.asc


## Scaling

In [5]:
SCALE_FACTOR = target_population / input_population

target_population_raster = np.where(valid_mask, input_population_raster * SCALE_FACTOR, nodata)

print(f"Sum of valid target population raster: {np.sum(target_population_raster[valid_mask]):,.0f}")

Sum of valid target population raster: 21,118,098


In [6]:
write_raster(    
    raster=target_population_raster,
    file=f"{generated_data_path}/{target_raster_name}",
    xllcorner=metadata["xllcorner"],
    yllcorner=metadata["yllcorner"],
    cellsize=metadata["cellsize"],
    nodata=nodata
)

print(f"Target population raster with population {np.sum(target_population_raster[valid_mask]):,.0f} written to: {generated_data_path}/{target_raster_name}")

Target population raster with population 21,118,098 written to: generated/ng-se_population_backwards_projected_2011_21.12M.asc


## Comparison

In [7]:
# correct_init_raster_path = "validation_1_0.25_population_scale_one_pattern_15_replicates/input/ng-se_initpopulation_2011_inferred_for_sim.asc"
# old_inferred_init_raster_path = "data/ng-se_initpopulation_2011.asc"

# incorrect_init_raster_path = "validation_6_0.25_population_scale_one_pattern_5_replicates/input/ng-se_initpopulation_2011_inferred_for_sim.asc"
# data_init_raster_path = f"{data_path}/{country_code}_initpopulation_{initial_year}_inferred_for_sim.asc"

# correct_init_raster, metadata = read_raster(correct_init_raster_path)
# incorrect_init_raster, _ = read_raster(incorrect_init_raster_path)
# data_init_raster, _ = read_raster(data_init_raster_path)
# observed_init_raster, _ = read_raster(old_inferred_init_raster_path)

# nodata = metadata["NODATA_value"]

# total_init_population_correct = np.sum(correct_init_raster[correct_init_raster != nodata])
# total_init_population_incorrect = np.sum(incorrect_init_raster[incorrect_init_raster != nodata])
# total_init_population_data = np.sum(data_init_raster[data_init_raster != nodata])
# total_init_population_old_inferred = np.sum(observed_init_raster[observed_init_raster != nodata])

# print(f"Total initial population (correct): {total_init_population_correct:,.0f}")
# print(f"Total initial population (incorrect): {total_init_population_incorrect:,.0f}")
# print(f"Total initial population (data): {total_init_population_data:,.0f}")
# print(f"Total initial population (old inferred): {total_init_population_old_inferred:,.0f}")

### Measure

In [8]:
raster_to_measure_path = "/home/tuv89272/work/Calibration_Pipeline_v7_NG-SE/validation_runs/validation_2_0.25_population_scale_one_pattern_20_replicates/input/ng-se_population_backwards_projected_2011_21.52M.asc" 

raster_to_measure, metadata = read_raster(raster_to_measure_path)

nodata = metadata["NODATA_value"]

total_population_raster_to_measure = np.sum(raster_to_measure[raster_to_measure != nodata])

print(f"Total population in raster to measure: {total_population_raster_to_measure:,.0f}")

Total population in raster to measure: 21,515,594


In [10]:
# raster_to_measure_path = ""

# raster_to_measure, metadata = read_raster(raster_to_measure_path)

# nodata = metadata["NODATA_value"]

# total_population_raster_to_measure = np.sum(raster_to_measure[raster_to_measure != nodata])

# print(f"Total population in raster to measure: {total_population_raster_to_measure:,.0f}")